# Source selection under a resource $\times$ proximity design

Follow-up experiment. The earlier pivot study confounded typological proximity
with source resource level: every alternative source was itself low-resource, so
a losing pivot could mean either that proximity does not help or just that the
source was data-poor. Here I break that confound. For each target I fine-tune
from sources chosen to cross two factors independently, and compare against the
English baseline:

- **high-resource close**: a well-resourced source that is also typologically
  near the target (the clean test of whether proximity helps once resources are
  controlled),
- **high-resource far**: well-resourced but typologically distant (control),
- **low-resource close**: near but data-poor (resembles the original pivot).

If high-resource-close beats English, proximity matters once resources are
equalised. If it still loses, proximity buys little even when resources are held
high, a stronger claim. Set `QUICK_SMOKE = True` to validate the pipeline first.

## Setup

In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" \
                "sentencepiece" "scikit-learn" "statsmodels" "lang2vec" 2>/dev/null

In [ ]:
import os, time, warnings, random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)
from sklearn.metrics import accuracy_score
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 42
set_seed(SEED); random.seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
USE_FP16 = DEVICE == "cuda" and not USE_BF16
print("device:", DEVICE, "| precision:", "bf16" if USE_BF16 else "fp16" if USE_FP16 else "fp32")

## Configuration

In [ ]:
QUICK_SMOKE       = False
XNLI_REPO         = "facebook/xnli"
OUTDIR            = "outputs_2x2"
os.makedirs(f"{OUTDIR}/figs", exist_ok=True)

# Two models: the cleanest encoder and the model where mitigation mattered most.
MODELS = {"xlmr": "xlm-roberta-base", "qwen": "Qwen/Qwen2.5-0.5B"}
TARGETS = ["ur","hi","el","ar","th","sw"]

# Source assignments from the resource x proximity analysis (CC-100 GiB, syn dist).
# cell -> {target: source}
SOURCES = {
    "hi_res_close": {"ur":"th","hi":"th","el":"fr","ar":"vi","th":"vi","sw":"th"},
    "hi_res_far":   {"ur":"de","hi":"de","el":"th","ar":"de","th":"de","sw":"de"},
    "lo_res_close": {"ur":"hi","hi":"ur","el":"hi","ar":"hi","th":"hi","sw":"tr"},
}

MAX_LEN, NUM_EPOCHS, LR = 128, 2, 2e-5
TRAIN_SAMPLE_SIZE, EVAL_SAMPLE_SIZE = 40000, 2000
BATCH = {"xlmr": 32, "qwen": 16}

if QUICK_SMOKE:
    MODELS = {"xlmr": "xlm-roberta-base"}
    TARGETS = ["ur","ar"]
    TRAIN_SAMPLE_SIZE, EVAL_SAMPLE_SIZE, NUM_EPOCHS = 800, 300, 1

import datasets as _ds
_orig = getattr(_ds, "_orig_load_dataset", _ds.load_dataset); _ds._orig_load_dataset = _orig
def load_dataset(path, *a, **k):
    if path == "xnli": path = XNLI_REPO
    return _orig(path, *a, **k)
_ds.load_dataset = load_dataset

# reference metadata for reporting
CC_GIB = {"ar":28.0,"bg":57.5,"de":66.6,"el":7.4,"es":53.3,"fr":56.8,"hi":20.2,
          "ru":278.0,"sw":1.6,"th":71.7,"tr":20.9,"ur":5.7,"vi":137.3,"zh":46.9,"en":300.8}
print("targets:", TARGETS)
for cell, mp in SOURCES.items():
    print(cell, "->", {t: f"{s}({CC_GIB[s]:.0f}G)" for t,s in mp.items() if t in TARGETS})

## Helpers

In [ ]:
def load_nli(lang, split, n):
    ds = load_dataset(XNLI_REPO, lang, split=split)
    return ds.shuffle(seed=SEED).select(range(n)) if n and n < len(ds) else ds

def make_tokenizer(name):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    return tok

def tokenize_ds(ds, tok):
    def f(b): return tok(b["premise"], b["hypothesis"], truncation=True, max_length=MAX_LEN)
    return ds.map(f, batched=True, remove_columns=[c for c in ds.column_names if c!="label"])

def build_model(name):
    m = AutoModelForSequenceClassification.from_pretrained(name, num_labels=3)
    if m.config.pad_token_id is None:
        m.config.pad_token_id = make_tokenizer(name).pad_token_id
    return m

def fine_tune(name, train_ds, tag):
    tok = make_tokenizer(name); model = build_model(name)
    args = TrainingArguments(
        output_dir=f"{OUTDIR}/ckpt_{tag}",
        per_device_train_batch_size=BATCH.get(tag.split("_")[0], 16),
        gradient_accumulation_steps=2, num_train_epochs=NUM_EPOCHS, learning_rate=LR,
        fp16=USE_FP16, bf16=USE_BF16, logging_steps=50, save_strategy="no",
        report_to="none", seed=SEED)
    tr = Trainer(model=model, args=args, train_dataset=tokenize_ds(train_ds, tok),
                 data_collator=DataCollatorWithPadding(tok))
    tr.train()
    return tr, tok

def evaluate_on(trainer, tok, lang, n):
    test = load_nli(lang, "test", n)
    pred = trainer.predict(tokenize_ds(test, tok))
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    return accuracy_score(np.array(test["label"]), logits.argmax(-1))*100

def free():
    if DEVICE == "cuda": torch.cuda.empty_cache()

## Run the 2x2

Each target is fine-tuned from three sources (high-res-close, high-res-far,
lo-res-close) and evaluated zero-shot. The English baseline is loaded from the
earlier run if present, else recomputed here.

In [ ]:
rows = []
for tag, mname in MODELS.items():
    # English baseline (recompute so this notebook is self-contained)
    print(f"\n[{tag}] English baseline")
    tr, tok = fine_tune(mname, load_nli("en","train",TRAIN_SAMPLE_SIZE), f"{tag}_en")
    for t in TARGETS:
        acc = evaluate_on(tr, tok, t, EVAL_SAMPLE_SIZE)
        rows.append({"model":tag,"target":t,"cell":"english","source":"en",
                     "src_gib":CC_GIB["en"],"acc":round(acc,2)})
        print(f"   {t}: {acc:.2f}")
    del tr; free()

    for cell, mp in SOURCES.items():
        for t in TARGETS:
            s = mp[t]
            print(f"[{tag}] {cell}: {t} <- {s}")
            tr, tok = fine_tune(mname, load_nli(s,"train",TRAIN_SAMPLE_SIZE), f"{tag}_{cell}_{t}")
            acc = evaluate_on(tr, tok, t, EVAL_SAMPLE_SIZE)
            rows.append({"model":tag,"target":t,"cell":cell,"source":s,
                         "src_gib":CC_GIB[s],"acc":round(acc,2)})
            print(f"   acc={acc:.2f}")
            del tr; free()

res = pd.DataFrame(rows); res.to_csv(f"{OUTDIR}/results_2x2.csv", index=False)
res.pivot_table(index=["model","target"], columns="cell", values="acc")

## Analysis: does proximity help once resources are controlled?

In [ ]:
res = pd.read_csv(f"{OUTDIR}/results_2x2.csv")
piv = res.pivot_table(index=["model","target"], columns="cell", values="acc").reset_index()

# deltas vs english baseline for each cell
for c in ["hi_res_close","hi_res_far","lo_res_close"]:
    piv[f"d_{c}"] = piv[c] - piv["english"]

print("=== mean delta vs English baseline (per model) ===")
agg = piv.groupby("model")[["d_hi_res_close","d_hi_res_far","d_lo_res_close"]].mean().round(2)
print(agg.to_string())

print("\n=== key contrasts (pooled over targets+models) ===")
from scipy import stats
def paired(a,b,label):
    a,b = np.array(a),np.array(b); d=a-b
    t,p = stats.ttest_rel(a,b)
    print(f"{label}: mean diff {d.mean():+.2f}, wins {sum(d>0)}/{len(d)}, t={t:.2f}, p={p:.4f}")

paired(piv.hi_res_close, piv.english,      "hi-res-close vs English   ")
paired(piv.hi_res_close, piv.lo_res_close, "hi-res-close vs lo-res-close (resource effect at fixed proximity)")
paired(piv.hi_res_close, piv.hi_res_far,   "hi-res-close vs hi-res-far  (proximity effect at fixed resource) ")
paired(piv.hi_res_far,  piv.english,       "hi-res-far  vs English    ")

piv.to_csv(f"{OUTDIR}/analysis_2x2.csv", index=False)
piv.round(2)

## Figure: the 2x2

In [ ]:
plt.rcParams.update({"font.family":"serif","font.size":9,"savefig.dpi":300})
piv = pd.read_csv(f"{OUTDIR}/analysis_2x2.csv")
cells = ["english","lo_res_close","hi_res_far","hi_res_close"]
labels = ["English\n(hi-res, far)","lo-res\nclose","hi-res\nfar","hi-res\nclose"]
fig, axes = plt.subplots(1, len(piv.model.unique()), figsize=(5.2,3),
                         sharey=True, squeeze=False)
for ax, m in zip(axes[0], piv.model.unique()):
    sub = piv[piv.model==m]
    means = [sub[c].mean() for c in cells]
    ax.bar(range(len(cells)), means,
           color=["#264653","#e76f51","#e9c46a","#2a9d8f"], edgecolor="k", linewidth=.4)
    ax.axhline(sub.english.mean(), color="0.5", lw=.7, ls="--")
    ax.set_xticks(range(len(cells))); ax.set_xticklabels(labels, fontsize=6.5)
    ax.set_title(m, fontsize=9)
    for s in ["top","right"]: ax.spines[s].set_visible(False)
axes[0][0].set_ylabel("zero-shot accuracy (%)")
plt.tight_layout(); plt.savefig(f"{OUTDIR}/figs/fig_2x2.png", bbox_inches="tight"); plt.show()